In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# 1. パラメータ設定
params_fixed = {
    'r_others': [1.5, 1.0, 1.0],    # [rx1, rx2, ry2] <- rx1を固定にする
    'lambda_11': 0.5,               # 固定
    'lambda_offdiag': [0.3, 0.3],   # [lambda12, lambda21] を固定値
    'c': [0.8, 0.6],                # [c1, c2]
    'd': [0.6, 0.8]                 # [d1, d2]
}

# ry1 はメッシュ変数になるので、ここで受け取るのは rx1 にする
rx1, rx2, ry2 = params_fixed['r_others'] 
l11 = params_fixed['lambda_11']
l12, l21 = params_fixed['lambda_offdiag']
c1, c2 = params_fixed['c']
d1, d2 = params_fixed['d']

# 2. メッシュグリッド
res = 400
l22_vals = np.linspace(0.001, 1.0, res) # X軸: l22
ry1_vals = np.linspace(0.001, 3.0, res) # Y軸: ry1
L22, RY1 = np.meshgrid(l22_vals, ry1_vals)

# 3. 計算
Det_x = c1 * d2 * l11 * L22 - c2 * d1 * l12 * l21
Det_y = l11 * L22 - l12 * l21

Num_x1 = d2 * L22 * rx1 - d1 * l21 * rx2
Num_x2 = c1 * l11 * rx2 - c2 * l12 * rx1
Num_y1 = L22 * RY1 - l12 * ry2      # ここにメッシュ変数 RY1 を使う
Num_y2 = l11 * ry2 - l21 * RY1      # ここにもメッシュ変数 RY1 を使う

with np.errstate(divide='ignore', invalid='ignore'):
    X1 = Num_x1 / Det_x
    X2 = Num_x2 / Det_x
    Y1 = Num_y1 / Det_y
    Y2 = Num_y2 / Det_y

# 4. 領域のカテゴリ分け
Category = np.zeros_like(X1, dtype=int)

pos_x1 = X1 > 0
pos_x2 = X2 > 0
pos_y1 = Y1 > 0
pos_y2 = Y2 > 0

# --- Step 1: 単独条件（背景） ---
Category[~pos_y1] = 2
Category[~pos_y2] = 3
Category[~pos_x1] = 4
Category[~pos_x2] = 5

# --- Step 2: 複合条件（優先度高） ---
mask_green = (~pos_y2) & (~pos_x2)
mask_gray  = (~pos_y1) & (~pos_x1)
mask_red   = (~pos_y2) & (~pos_x1)
mask_blue  = (~pos_y1) & (~pos_x2)

Category[mask_green] = 6
Category[mask_gray]  = 7
Category[mask_red]   = 8
Category[mask_blue]  = 9

# --- Step 3: 共存（最優先） ---
mask_coexist = pos_x1 & pos_x2 & pos_y1 & pos_y2
Category[mask_coexist] = 1

# 5. プロット
fig, ax = plt.subplots(figsize=(12, 8))

colors = [
    'white', 'gold', 'skyblue', 'salmon', 'SlateBlue', 
    'IndianRed', 'MediumSeaGreen', 'lightgray', 'red', 'blue'
]
cmap = ListedColormap(colors)
norm = BoundaryNorm(np.arange(-0.5, 10.5, 1), cmap.N)

im = ax.imshow(Category, extent=[0, 1.0, 0, 3.0], origin='lower',
               cmap=cmap, norm=norm, aspect='auto', alpha=0.9)

# --- 補助線 ---
# Det_y = 0 line
singularity_y_val = (l12 * l21) / l11

# Det_x = 0 line
num_sing_x = c2 * d1 * l12 * l21
den_sing_x = c1 * d2 * l11
singularity_x_val = num_sing_x / den_sing_x

# Num_y1 = 0 の境界線 (今回はY軸がry1なので、これが適切です)
# Num_y1 = L22 * RY1 - l12 * ry2 = 0  =>  RY1 = (l12 * ry2) / L22
x_line = np.linspace(0.001, 1.0, 100)
y_line_num_y1 = (l12 * ry2) / x_line
y_line_num_y1[y_line_num_y1 > 3.0] = np.nan # 範囲外をクリップ

ax.axvline(x=singularity_y_val, color='white', linestyle='-', linewidth=3, label=r'Singularity ($Det_y = 0$)')
ax.axvline(x=singularity_x_val, color='black', linestyle='--', linewidth=3, label=r'Singularity ($Det_x = 0$)')
ax.plot(x_line, y_line_num_y1, color='purple', linestyle=':', linewidth=2, label=r'$Num_{y1}=0$ Boundary')

# --- ラベル設定 ---
ax.set_xlabel(r'Intraspecific Competition $\lambda_{22}$')
ax.set_ylabel(r'Intrinsic Growth Rate $r_{y1}$')
ax.set_title(r'Phase Diagram ($\lambda_{22}$ vs $r_{y1}$)')
ax.set_xlim(0, 1.0)
ax.set_ylim(0, 3.0)

# テキスト表示 (ry1を外してrx1を表示する)
textstr = '\n'.join((
    r'Fixed Parameters:',
    r'$\lambda_{11} = %.1f$' % l11,
    r'$\lambda_{12}=%.1f, \lambda_{21}=%.1f$' % (l12, l21),
    r'$r_{others} = [%.1f, %.1f, %.1f]$' % (rx1, rx2, ry2), # rx1に変更
    r'$c=[%.1f, %.1f], d=[%.1f, %.1f]$' % (c1, c2, d1, d2)
))
props = dict(boxstyle='round', facecolor='white', alpha=0.9)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=props)

# --- 凡例設定 ---
legend_elements = [
    Patch(facecolor='gold', edgecolor='k', label='Coexistence'),
    Patch(facecolor='MediumSeaGreen', edgecolor='k', label='$y_2 < 0$ & $x_2 < 0$ (Green)'),
    Patch(facecolor='lightgray', edgecolor='k', label='$y_1 < 0$ & $x_1 < 0$ (Gray)'),
    Patch(facecolor='red', edgecolor='k', label='$y_2 < 0$ & $x_1 < 0$ (Red)'),
    Patch(facecolor='blue', edgecolor='k', label='$y_1 < 0$ & $x_2 < 0$ (Blue)'),
    Patch(facecolor='skyblue', edgecolor='k', label='Only $y_1 < 0$'),
    Patch(facecolor='salmon', edgecolor='k', label='Only $y_2 < 0$'),
    Patch(facecolor='SlateBlue', edgecolor='k', label='Only $x_1 < 0$'),
    Patch(facecolor='IndianRed', edgecolor='k', label='Only $x_2 < 0$'),
    
    Line2D([0], [0], color='white', lw=3, label=r'$Det_y=0$'),
    Line2D([0], [0], color='black', linestyle='--', lw=3, label=r'$Det_x=0$'),
    Line2D([0], [0], color='purple', linestyle=':', lw=2, label=r'$Num_{y1}=0$') # ラベル変更
]

plt.subplots_adjust(right=0.7)
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1.0), borderaxespad=0.)
# 出力ディレクトリ設定
output_dir = Path("../phase_diagram_images")
output_dir.mkdir(exist_ok=True, parents=True)

# 画像を保存
output_path = output_dir / "ry1_l22.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.close(fig)

plt.show()